# Age Uncertainty Assignment

Assigns `age_err_lo_gyr` and `age_err_hi_gyr` to the 2456 cfrc stars in `training_stars.csv` that currently have NaN age errors.

Two sources, in priority order:
1. **MOCADB cluster/association age uncertainty** (`age_myr_unc_pos/neg`) — for cfrc subgroups mapped to a MOCADB moca_aid with a formal uncertainty
2. **cluster_properties `logA_err`** — std dev of log age across the 4 ChronoFlow age catalogs (G+18, B+19, CG+20, L+23), converted to asymmetric Myr uncertainties

Stars with neither source keep NaN (272 stars: ROph_L1688, Taurus subgroups, M37, HPer, small MOCADB comoving groups).

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = 'cf_data/'

training  = pd.read_csv(DATA_DIR + 'training_stars.csv')
cfrc      = pd.read_csv(DATA_DIR + 'cf_rotator_catalog.csv')
mocadb_cf = pd.read_csv(DATA_DIR + 'mocadb.csv')
cp        = pd.read_csv('reference/cluster_properties.csv')

print(f'training_stars: {len(training)} rows')
print(f'Stars with NaN age_err_lo_gyr: {training["age_err_lo_gyr"].isna().sum()}')

training_stars: 6119 rows
Stars with NaN age_err_lo_gyr: 2456


## Source 1 — MOCADB cluster/association age uncertainties

Map each cfrc subgroup to its MOCADB moca_aid, then look up `age_myr_unc_pos/neg` from `cf_data/mocadb.csv`.

In [2]:
# cfrc subgroup → MOCADB moca_aid mapping
cfrc_to_mocadb = {
    'Pleiades':     'PLE',
    'Praesepe':     'PRA',
    'Hyades':       'HYA',
    'APer':         'APER',
    'NGC2516':      'NGC2516',
    'IC2391':       'IC2391',
    'IC2602':       'IC2602',
    'NGC2547':      'NGC2547',
    'Collinder135': 'COL135',
    'NGC3532':      'NGC3532',
    'NGC2451A':     'NGC2451A',
    'Ruprecht147':  'RUP147',
    'USco_fg':      'USCO',
    'NGC6811':      'NGC6811',
    'DSco':         'GRSCO',
    'SSco':         'GRSCO',
    'BSco':         'GRSCO',
    'NSco':         'GRSCO',
    'Antares':      'GRSCO',
    'RSco':         'GRSCO',
    'ScoBody':      'GRSCO',
}

# Pull cluster-level age uncertainties from mocadb (one row per moca_aid)
mocadb_cluster_unc = (
    mocadb_cf[mocadb_cf['age_myr_unc_pos'].notna()]
    .groupby('moca_aid')[['age_myr_unc_pos', 'age_myr_unc_neg']]
    .first()
)

print('MOCADB cluster uncertainties for mapped subgroups (Myr):')
for sg, maid in sorted(cfrc_to_mocadb.items()):
    if maid in mocadb_cluster_unc.index:
        row = mocadb_cluster_unc.loc[maid]
        print(f'  {sg:15s} → {maid:10s}: +{row["age_myr_unc_pos"]:.1f}/-{row["age_myr_unc_neg"]:.1f} Myr')
    else:
        print(f'  {sg:15s} → {maid:10s}: NOT FOUND')

MOCADB cluster uncertainties for mapped subgroups (Myr):
  APer            → APER      : +1.5/-2.3 Myr
  Antares         → GRSCO     : +25.0/-10.0 Myr
  BSco            → GRSCO     : +25.0/-10.0 Myr
  Collinder135    → COL135    : +5.5/-5.5 Myr
  DSco            → GRSCO     : +25.0/-10.0 Myr
  Hyades          → HYA       : +85.0/-67.0 Myr
  IC2391          → IC2391    : +5.6/-3.9 Myr
  IC2602          → IC2602    : +1.9/-1.4 Myr
  NGC2451A        → NGC2451A  : +7.3/-7.3 Myr
  NGC2516         → NGC2516   : +25.0/-25.0 Myr
  NGC2547         → NGC2547   : +1.2/-1.2 Myr
  NGC3532         → NGC3532   : +20.0/-20.0 Myr
  NGC6811         → NGC6811   : +200.0/-200.0 Myr
  NSco            → GRSCO     : +25.0/-10.0 Myr
  Pleiades        → PLE       : +6.3/-10.0 Myr
  Praesepe        → PRA       : +40.0/-10.0 Myr
  RSco            → GRSCO     : +25.0/-10.0 Myr
  Ruprecht147     → RUP147    : +618.1/-618.1 Myr
  SSco            → GRSCO     : +25.0/-10.0 Myr
  ScoBody         → GRSCO     : +25.0/-1

## Source 2 — cluster_properties logA_err

`logA_err` is the std dev of log10(age/yr) across the 4 ChronoFlow age catalogs.
Converts to asymmetric Myr uncertainties:
- `age_err_hi_Myr = age_Myr * (10^logA_err - 1)`
- `age_err_lo_Myr = age_Myr * (1 - 10^(-logA_err))`

In [3]:
# Build cluster_properties lookup: cluster name + altname → (age_Myr, logA_err)
cp_lookup = {}
for _, row in cp.iterrows():
    if pd.notna(row['logA_err']):
        cp_lookup[row['cluster']] = (row['age_Myr'], row['logA_err'])
        if pd.notna(row['cluster_altname']) and row['cluster_altname'] != 'NULL':
            cp_lookup[row['cluster_altname']] = (row['age_Myr'], row['logA_err'])

def logA_err_to_myr(age_Myr, logA_err):
    err_hi = age_Myr * (10**logA_err - 1)
    err_lo = age_Myr * (1 - 10**(-logA_err))
    return err_hi, err_lo

print('cluster_properties coverage (clusters with logA_err):')
for cl, (age, logA_err) in sorted(cp_lookup.items()):
    err_hi, err_lo = logA_err_to_myr(age, logA_err)
    print(f'  {cl:15s}: {age:.0f} Myr, logA_err={logA_err:.4f} → +{err_hi:.0f}/-{err_lo:.0f} Myr')

cluster_properties coverage (clusters with logA_err):
  APer           : 68 Myr, logA_err=0.1150 → +21/-16 Myr
  Hyades         : 775 Myr, logA_err=0.0144 → +26/-25 Myr
  IC2391         : 37 Myr, logA_err=0.1205 → +12/-9 Myr
  IC2602         : 37 Myr, logA_err=0.0276 → +2/-2 Myr
  M34            : 161 Myr, logA_err=0.1674 → +76/-51 Myr
  M35            : 188 Myr, logA_err=0.2923 → +181/-92 Myr
  M50            : 135 Myr, logA_err=0.1628 → +61/-42 Myr
  M67            : 3870 Myr, logA_err=0.0440 → +413/-373 Myr
  Melotte20      : 68 Myr, logA_err=0.1150 → +21/-16 Myr
  Melotte22      : 97 Myr, logA_err=0.0918 → +23/-18 Myr
  NGC1039        : 161 Myr, logA_err=0.1674 → +76/-51 Myr
  NGC1647        : 309 Myr, logA_err=0.0990 → +79/-63 Myr
  NGC1750        : 322 Myr, logA_err=0.1038 → +87/-68 Myr
  NGC1758        : 436 Myr, logA_err=0.1734 → +214/-144 Myr
  NGC1817        : 1273 Myr, logA_err=0.0777 → +249/-209 Myr
  NGC2168        : 188 Myr, logA_err=0.2923 → +181/-92 Myr
  NGC2281       

## Assign age uncertainties to cfrc stars

In [4]:
# Identify cfrc stars (NaN age errors) and merge subgroup from cfrc catalog
cfrc_mask = training['age_err_lo_gyr'].isna() & training['age_err_hi_gyr'].isna()
cfrc_idx  = training[cfrc_mask].index

# Merge subgroup from cfrc catalog
subgroup_map = cfrc.set_index('GaiaDR3_ID')['subgroup']
training.loc[cfrc_idx, 'subgroup_tmp'] = (
    training.loc[cfrc_idx, 'source_id']
    .map(lambda x: subgroup_map.get(int(x)) if pd.notna(x) else None)
)

# Merge moca_aid for no-subgroup stars
moca_map = mocadb_cf.set_index('GaiaDR3_ID')['moca_aid']
training.loc[cfrc_idx, 'moca_aid_tmp'] = (
    training.loc[cfrc_idx, 'source_id']
    .map(lambda x: moca_map.get(int(x)) if pd.notna(x) else None)
)

n_source1, n_source2, n_none = 0, 0, 0

for idx in cfrc_idx:
    row      = training.loc[idx]
    age_Myr  = row['age_gyr'] * 1000
    subgroup = row.get('subgroup_tmp')
    moca_aid = row.get('moca_aid_tmp')

    err_hi_gyr, err_lo_gyr = None, None

    # Source 1: MOCADB cluster uncertainty via subgroup mapping
    if pd.notna(subgroup) and subgroup in cfrc_to_mocadb:
        maid = cfrc_to_mocadb[subgroup]
        if maid in mocadb_cluster_unc.index:
            unc = mocadb_cluster_unc.loc[maid]
            err_hi_gyr = unc['age_myr_unc_pos'] / 1000
            err_lo_gyr = unc['age_myr_unc_neg'] / 1000
            n_source1 += 1

    # Source 2: cluster_properties logA_err — try subgroup name first, then moca_aid
    if err_hi_gyr is None:
        key = subgroup if pd.notna(subgroup) and subgroup in cp_lookup else (
              moca_aid if pd.notna(moca_aid) and moca_aid in cp_lookup else None)
        if key is not None:
            cp_age, logA_err = cp_lookup[key]
            err_hi, err_lo = logA_err_to_myr(age_Myr, logA_err)
            err_hi_gyr = err_hi / 1000
            err_lo_gyr = err_lo / 1000
            n_source2 += 1

    if err_hi_gyr is None:
        n_none += 1

    training.loc[idx, 'age_err_hi_gyr'] = err_hi_gyr
    training.loc[idx, 'age_err_lo_gyr'] = err_lo_gyr

# Drop temp columns
training.drop(columns=['subgroup_tmp', 'moca_aid_tmp'], inplace=True)

print(f'Assigned from MOCADB cluster uncertainty: {n_source1}')
print(f'Assigned from cluster_properties logA_err: {n_source2}')
print(f'No uncertainty found (remain NaN):          {n_none}')
print(f'Total cfrc stars processed: {n_source1 + n_source2 + n_none}')

Assigned from MOCADB cluster uncertainty: 1674
Assigned from cluster_properties logA_err: 509
No uncertainty found (remain NaN):          273
Total cfrc stars processed: 2456


## Diagnostics

In [5]:
still_nan = training['age_err_lo_gyr'].isna().sum()
print(f'Stars with NaN age_err after assignment: {still_nan} / {len(training)}')
print(f'Stars with age_err filled: {len(training) - still_nan} / {len(training)}')
print()

# Sanity check: sample one star per source
filled = training[training['age_err_lo_gyr'].notna()]
print('Sample (age_gyr, age_err_lo_gyr, age_err_hi_gyr):')
print(filled[['age_gyr', 'age_err_lo_gyr', 'age_err_hi_gyr']].describe().round(4))

Stars with NaN age_err after assignment: 273 / 6119
Stars with age_err filled: 5846 / 6119

Sample (age_gyr, age_err_lo_gyr, age_err_hi_gyr):
         age_gyr  age_err_lo_gyr  age_err_hi_gyr
count  5846.0000       5846.0000       5846.0000
mean      0.4054          0.0645          0.1055
std       0.9387          0.2398          0.4358
min       0.0015          0.0003          0.0003
25%       0.0450          0.0100          0.0063
50%       0.1330          0.0100          0.0250
75%       0.3980          0.0450          0.0614
max      11.5000          4.9790          8.6000


## Save

In [6]:
training.to_csv(DATA_DIR + 'training_stars.csv', index=False)
print('Saved training_stars.csv')

Saved training_stars.csv
